# Proyecto: Predicción de Incumplimiento de SLA en Sistema de Soporte Técnico
## Fase 1: Recopilación y Transformación de Datos

**Fuente de datos:** Base de datos Supabase - Sistema Lillosupport  
**Tablas utilizadas:** `incidencias`, `ans_resultados`, `ans_configuracion`, `tipos_servicio`  
**Objetivo:** Preparar un dataset limpio y enriquecido para predecir si una incidencia cumplirá o no el ANS.

## Carga del Dataset



In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Cargar el dataset
df = pd.read_csv('BD.csv')

print(f'Columnas: {df.shape[1]}')
df.head()

In [ ]:
# Resumen general del dataset
print('--Tipos de datos')
print(df.dtypes)
print()
print('--Valores nulos por columna')
print(df.isnull().sum())

## Limpieza de las columnas

In [ ]:
# Columnas de tipo fecha
columnas_fecha = [
    'created_at', 'updated_at', 'fecha_creacion',
    'fecha_limite_ans', 'ans_fecha_limite', 'ans_fecha_cierre'
]

for col in columnas_fecha:
    df[col] = pd.to_datetime(df[col], utc=True, errors='coerce')

print('Fechas convertidas correctamente.')
df[columnas_fecha].dtypes

### Creación de la variable objetivo: `cumple_ans`

Esta es la variable que el modelo predictivo intentará predecir.  
- `1` Significaque la incidencia fue resuelta antes del límite de tiempo (cumplió el ANS)  
- `0` Significa que la incidencia superó el tiempo límite (incumplió el ANS)

In [ ]:
# Variable objetivo basada en ans_estado_final
df['cumple_ans'] = df['ans_estado_final'].apply(
    lambda x: 1 if str(x).strip().lower() == 'cumplido' else 0
)

print('Distribución de la variable objetivo:')
print(df['cumple_ans'].value_counts())
print()
print(f"Tasa de cumplimiento: {df['cumple_ans'].mean()*100:.1f}%")

### Creación de variables derivadas

In [ ]:
# Tiempo real de resolución en horas (desde creación hasta cierre)
df['horas_resolucion'] = (
    df['ans_fecha_cierre'] - df['fecha_creacion']
).dt.total_seconds() / 3600

# Tiempo límite en horas (desde creación hasta límite ANS)
df['horas_limite_ans'] = (
    df['ans_fecha_limite'] - df['fecha_creacion']
).dt.total_seconds() / 3600

# Horas de diferencia entre cierre y límite que signifia negativo = cumplió, positivo = venció
df['horas_diferencia_ans'] = (
    df['ans_fecha_cierre'] - df['ans_fecha_limite']
).dt.total_seconds() / 3600

# Porcentaje de tiempo consumido respecto al total
df['pct_tiempo_consumido'] = (
    df['minutos_consumidos'] / df['minutos_totales'].replace(0, np.nan)
) * 100

# Variables de fecha: mes, día de semana, hora de creación
df['mes_creacion'] = df['fecha_creacion'].dt.month
df['dia_semana_creacion'] = df['fecha_creacion'].dt.dayofweek  # 0=Lunes, 6=Domingo
df['hora_creacion'] = df['fecha_creacion'].dt.hour

# Turno del día
def turno(hora):
    if 6 <= hora < 12:
        return 'mañana'
    elif 12 <= hora < 18:
        return 'tarde'
    elif 18 <= hora < 24:
        return 'noche'
    else:
        return 'madrugada'

df['turno_creacion'] = df['hora_creacion'].apply(turno)

print('Variables derivadas creadas correctamente.')
df[['horas_resolucion','horas_limite_ans','horas_diferencia_ans',
    'pct_tiempo_consumido','mes_creacion','dia_semana_creacion',
    'hora_creacion','turno_creacion']].head()

### Limpieza de columnas

In [ ]:
# Limpiar columna ans_tiempo_limite pasar de '5d' a 5
df['dias_limite_config'] = df['ans_tiempo_limite'].str.extract(r'(\d+)').astype(float)

# Estandarizar texto en columnas categóricas
for col in ['estado', 'prioridad', 'tipo_servicio', 'ans_estado', 'ans_estado_final', 'turno_creacion']:
    df[col] = df[col].str.strip().str.lower()

# Columna finalizado a booleano numérico
df['finalizado'] = df['finalizado'].map({'true': 1, 'false': 0, True: 1, False: 0})

print('Limpieza completada.')
print(f'Valores nulos restantes:')
print(df.isnull().sum()[df.isnull().sum() > 0])

### Tratamiento de valores nulos

In [ ]:
# Rellenar nulos en columnas numéricas con la mediana
cols_numericas = ['horas_resolucion', 'horas_limite_ans', 'horas_diferencia_ans',
                  'pct_tiempo_consumido', 'costo_base', 'descuento_ans']

# Guardar máscara antes de rellenar
mask_nulos = df['ans_fecha_cierre'].isnull()

for col in cols_numericas:
    if df[col].isnull().sum() > 0:
        mediana = df[col].median()
        df[col].fillna(mediana, inplace=True)
        print(f'{col}: rellenado con mediana ({mediana:.2f})')

print('\n--Nulos despues del proceo')
for col in cols_numericas:
    print(f'{col}: {df[col].isnull().sum()} nulos restantes')

print(f'\n--Filas que fueron rellenadas: {mask_nulos.sum()}')
df[mask_nulos][['incidencia_id', 'horas_resolucion', 'horas_limite_ans',
                'horas_diferencia_ans', 'pct_tiempo_consumido']]

## Dataset final limpio

In [ ]:
# Resultado
columnas_finales = [
    'incidencia_id',
    'fecha_creacion',
    'estado',
    'prioridad',
    'tipo_servicio',
    'ans_estado',
    'ans_estado_final',
    'cumple_ans',
    'horas_resolucion',
    'horas_limite_ans',
    'horas_diferencia_ans',
    'pct_tiempo_consumido',
    'minutos_consumidos',
    'minutos_totales',
    'minutos_restantes',
    'dias_limite_config',
    'ans_penalidad_porcentaje',
    'costo_base_servicio',
    'penalidad_porcentaje',
    'costo_servicio',
    'mes_creacion',
    'dia_semana_creacion',
    'hora_creacion',
    'turno_creacion',
    'finalizado',
    'billing_status'
]

df_final = df[columnas_finales].copy()

# Guardar dataset limpio
df_final.to_csv('BD_limpia.csv', index=False)

print('=== DATASET FINAL ===')
print(f'Registros: {df_final.shape[0]}')
print(f'Columnas: {df_final.shape[1]}')
print(f'Cumple ANS: {df_final["cumple_ans"].sum()} ({df_final["cumple_ans"].mean()*100:.1f}%)')
print(f'Incumple ANS: {(df_final["cumple_ans"]==0).sum()} ({(1-df_final["cumple_ans"].mean())*100:.1f}%)')
print()
df_final.head(10)

In [ ]:
# Estadísticas descriptivas del dataset final
df_final.describe()

## Fase 2: Análisis exploratorio de datos

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo de gráficas
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

df = pd.read_csv('BD_limpia.csv', parse_dates=['fecha_creacion'])

print(f'Registros: {df.shape[0]} | Columnas: {df.shape[1]}')
df.head()

## Variable Objetivo: cumple_ans


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de barras
conteo = df['cumple_ans'].value_counts()
etiquetas = ['Incumplió ANS\n(0)', 'Cumplió ANS\n(1)']
colores = ['#e74c3c', '#2ecc71']
axes[0].bar(etiquetas, [conteo[0], conteo[1]], color=colores, edgecolor='black', width=0.5)
axes[0].set_title('Distribución de Cumplimiento de ANS', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cantidad de Incidencias')
for i, v in enumerate([conteo[0], conteo[1]]):
    axes[0].text(i, v + 0.5, f'{v} ({v}%)', ha='center', fontweight='bold')

# Gráfico de torta
axes[1].pie([conteo[0], conteo[1]], labels=['Incumplió\nANS', 'Cumplió\nANS'],
            colors=colores, autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12})
axes[1].set_title('Proporción de Cumplimiento', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_cumple_ans.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dataset perfectamente balanceado: 50% cumple / 50% incumple')